# 02 — Pré-processamento e separação treino/teste

Este notebook documenta as decisões de preparação do Iranian Churn Dataset e cria uma separação reprodutível entre treino e teste sem permitir que perfis idênticos apareçam nos dois conjuntos.

In [1]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler

In [2]:
df = pd.read_csv("../data/raw/iranian_churn.csv")

# Os nomes originais contêm espaços simples e duplos. A normalização também
# mantém os campos compatíveis com os nomes usados pelo restante do projeto.
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

df.head()

,call_failure,complains,subscription_length,charge_amount,seconds_of_use,frequency_of_use,frequency_of_sms,distinct_called_numbers,age_group,tariff_plan,status,age,customer_value,churn
0,8,0,38,0,4370,71,5,17,3,1,1,30,197.640,0
1,0,0,39,0,318,5,7,4,2,1,2,25,46.035,0
2,10,0,37,0,2453,60,359,24,3,1,1,30,1536.520,0
3,10,0,38,0,4198,66,1,35,1,1,1,15,240.020,0
4,3,0,38,0,2393,58,2,33,1,1,1,15,145.805,0


In [3]:
print(f"Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")
print("\nTipos:")
print(df.dtypes)
print(f"\nTotal de valores nulos: {df.isna().sum().sum()}")
print(f"Classes do alvo: {sorted(df['churn'].unique().tolist())}")
print(f"Intervalo de charge_amount: {df['charge_amount'].min()} a {df['charge_amount'].max()}")
print(f"Intervalo de frequency_of_sms: {df['frequency_of_sms'].min()} a {df['frequency_of_sms'].max()}")

assert df.shape == (3150, 14)
assert df.isna().sum().sum() == 0
assert set(df['churn'].unique()) == {0, 1}
assert df['charge_amount'].between(0, 10).all()

Dimensões: 3150 linhas x 14 colunas

Tipos:
call_failure                 int64
complains                    int64
subscription_length          int64
charge_amount                int64
seconds_of_use               int64
frequency_of_use             int64
frequency_of_sms             int64
distinct_called_numbers      int64
age_group                    int64
tariff_plan                  int64
status                       int64
age                          int64
customer_value             float64
churn                        int64
dtype: object

Total de valores nulos: 0
Classes do alvo: [0, 1]
Intervalo de charge_amount: 0 a 10
Intervalo de frequency_of_sms: 0 a 522


## Decisões de tratamento

- Não há valores nulos; portanto, não será aplicada imputação.
- As linhas repetidas serão preservadas, pois podem representar clientes distintos com o mesmo perfil.
- Os valores extremos de `charge_amount` e `frequency_of_sms` são plausíveis no contexto do dataset e não serão removidos, limitados ou winsorizados.
- As repetições serão usadas apenas para formar grupos durante a separação treino/teste, impedindo vazamento entre os conjuntos.
- `age_group` será retirada das variáveis do modelo porque é completamente determinada por `age` neste dataset.

In [4]:
preditores_originais = df.drop(columns="churn")

duplicatas_completas_excedentes = int(df.duplicated().sum())
linhas_em_grupos_completos_repetidos = int(df.duplicated(keep=False).sum())
duplicatas_preditores_excedentes = int(preditores_originais.duplicated().sum())
linhas_em_grupos_de_preditores_repetidos = int(
    preditores_originais.duplicated(keep=False).sum()
)

print(f"Cópias completas excedentes: {duplicatas_completas_excedentes}")
print(f"Linhas em grupos completos repetidos: {linhas_em_grupos_completos_repetidos}")
print(f"Cópias excedentes considerando apenas preditores: {duplicatas_preditores_excedentes}")
print(f"Linhas em grupos de preditores repetidos: {linhas_em_grupos_de_preditores_repetidos}")

assert duplicatas_completas_excedentes == 300
assert len(df) == 3150  # nenhuma repetição foi removida

Cópias completas excedentes: 300
Linhas em grupos completos repetidos: 465
Cópias excedentes considerando apenas preditores: 314
Linhas em grupos de preditores repetidos: 476


In [5]:
relacao_idade_faixa = pd.crosstab(df["age"], df["age_group"])
display(relacao_idade_faixa)

# Cada idade aparece em uma única faixa, logo age_group não acrescenta
# informação ao modelo neste dataset.
assert (preditores_originais.groupby("age")["age_group"].nunique() == 1).all()

X = preditores_originais.drop(columns="age_group").copy()
y = df["churn"].copy()

# Encoding binário explícito. Como cada variável possui somente duas categorias,
# uma coluna 0/1 preserva toda a informação sem criar colunas redundantes.
X["tariff_plan"] = X["tariff_plan"].map({1: 0, 2: 1})
X["status"] = X["status"].map({1: 0, 2: 1})

assert set(X["complains"].unique()) <= {0, 1}
assert set(X["tariff_plan"].unique()) <= {0, 1}
assert set(X["status"].unique()) <= {0, 1}

print(f"Preditores usados pelo modelo ({X.shape[1]}):")
print(X.columns.tolist())
print("\nEncoding: complains já era 0/1; tariff_plan e status foram convertidos de 1/2 para 0/1.")

age_group,1,2,3,4,5
age,,,,,
15,123,0,0,0,0
25,0,1037,0,0,0
30,0,0,1425,0,0
45,0,0,0,395,0
55,0,0,0,0,170


Preditores usados pelo modelo (12):
['call_failure', 'complains', 'subscription_length', 'charge_amount', 'seconds_of_use', 'frequency_of_use', 'frequency_of_sms', 'distinct_called_numbers', 'tariff_plan', 'status', 'age', 'customer_value']

Encoding: complains já era 0/1; tariff_plan e status foram convertidos de 1/2 para 0/1.


### Escolha do encoding

`complains`, `tariff_plan` e `status` são variáveis binárias. Por isso foi adotada uma codificação explícita em uma única coluna 0/1, equivalente ao label encoding para duas categorias. One-hot encoding criaria colunas redundantes e não acrescentaria informação. `charge_amount` foi mantida como variável ordinal, pois seus valores representam faixas ordenadas de cobrança.

In [6]:
# O agrupamento usa todos os preditores originais, antes da retirada de
# age_group. O alvo não participa da identificação do perfil.
grupos, perfis_unicos = pd.factorize(
    pd.MultiIndex.from_frame(preditores_originais),
    sort=False,
)

alvos_por_grupo = (
    pd.DataFrame({"grupo": grupos, "churn": y})
    .groupby("grupo")["churn"]
    .agg(tamanho="size", quantidade_de_alvos="nunique")
)
grupos_com_alvos_divergentes = alvos_por_grupo.query("quantidade_de_alvos > 1")

print(f"Perfis preditores únicos: {len(perfis_unicos)}")
print(f"Grupos com alvos divergentes: {len(grupos_com_alvos_divergentes)}")
print(f"Linhas nesses grupos: {grupos_com_alvos_divergentes['tamanho'].sum()}")

assert len(perfis_unicos) == 2836

Perfis preditores únicos: 2836
Grupos com alvos divergentes: 14
Linhas nesses grupos: 64


In [7]:
separador = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

indice_treino, indice_teste = next(separador.split(X, y, groups=grupos))

X_train = X.iloc[indice_treino].copy()
X_test = X.iloc[indice_teste].copy()
y_train = y.iloc[indice_treino].copy()
y_test = y.iloc[indice_teste].copy()
grupos_train = grupos[indice_treino].copy()
grupos_test = grupos[indice_teste].copy()
ids_grupos_treino = set(grupos_train)
ids_grupos_teste = set(grupos_test)

assert ids_grupos_treino.isdisjoint(ids_grupos_teste)
assert len(X_train) + len(X_test) == len(df)
assert X_train.index.intersection(X_test.index).empty
assert abs(y_train.mean() - y_test.mean()) < 0.01

In [8]:
resumo_separacao = pd.DataFrame({
    "conjunto": ["treino", "teste", "total"],
    "registros": [len(X_train), len(X_test), len(df)],
    "proporcao_churn": [y_train.mean(), y_test.mean(), y.mean()],
})

display(resumo_separacao.style.format({"proporcao_churn": "{:.2%}"}))
print(f"Perfis compartilhados entre treino e teste: {len(ids_grupos_treino & ids_grupos_teste)}")

,conjunto,registros,proporcao_churn
0,treino,2517,15.69%
1,teste,633,15.80%
2,total,3150,15.71%


Perfis compartilhados entre treino e teste: 0


In [9]:
contagem_classes = y.value_counts().sort_index()
distribuicao_classes = pd.DataFrame({
    "classe": ["0 — não churn", "1 — churn"],
    "registros": [contagem_classes[0], contagem_classes[1]],
    "proporcao": [contagem_classes[0] / len(y), contagem_classes[1] / len(y)],
})
razao_desbalanceamento = contagem_classes[0] / contagem_classes[1]
acuracia_baseline_majoritaria = contagem_classes[0] / len(y)

display(distribuicao_classes.style.format({"proporcao": "{:.2%}"}))
print(f"Razão entre não churn e churn: {razao_desbalanceamento:.2f}:1")
print(f"Acurácia de sempre prever não churn: {acuracia_baseline_majoritaria:.2%}")

,classe,registros,proporcao
0,0 — não churn,2655,84.29%
1,1 — churn,495,15.71%


Razão entre não churn e churn: 5.36:1
Acurácia de sempre prever não churn: 84.29%


## Desbalanceamento e limitação da acurácia

A classe sem churn representa 84,29% dos registros. Portanto, um classificador ingênuo que previsse "não churn" para todos os clientes obteria 84,29% de acurácia, mas recall igual a zero para churn: não identificaria corretamente nenhum cliente propenso ao cancelamento. A acurácia será apresentada, porém não será usada isoladamente para escolher o modelo. A avaliação deverá priorizar precisão, recall e F1-score da classe churn, além das médias macro e ponderada e da matriz de confusão.

As estratégias previstas para comparação futura são: baseline sem balanceamento, SMOTE e subamostragem aleatória (undersampling). Elas serão aplicadas somente aos dados de treino e dentro dos pipelines de validação, nunca antes da separação treino/teste.

## Preparação das transformações

Padronização e normalização serão comparadas para SVM e KNN, que são sensíveis à escala. Random Forest pode receber os dados sem escala porque árvores trabalham com divisões e não com distâncias. Os transformadores destinados aos modelos permanecem sem ajuste: seu `fit` deverá ocorrer dentro do pipeline e somente com os dados de treino.

In [10]:
colunas_binarias = ["complains", "tariff_plan", "status"]
colunas_numericas = [
    coluna for coluna in X.columns if coluna not in colunas_binarias
]

preprocessador_standard = ColumnTransformer(
    transformers=[
        ("escala", StandardScaler(), colunas_numericas),
        ("binarias", "passthrough", colunas_binarias),
    ],
    verbose_feature_names_out=False,
)

preprocessador_minmax = ColumnTransformer(
    transformers=[
        ("escala", MinMaxScaler(), colunas_numericas),
        ("binarias", "passthrough", colunas_binarias),
    ],
    verbose_feature_names_out=False,
)

print("Transformadores definidos, mas ainda não ajustados.")
print("Colunas que serão escalonadas:", colunas_numericas)
print("Colunas binárias preservadas em 0/1:", colunas_binarias)

Transformadores definidos, mas ainda não ajustados.
Colunas que serão escalonadas: ['call_failure', 'subscription_length', 'charge_amount', 'seconds_of_use', 'frequency_of_use', 'frequency_of_sms', 'distinct_called_numbers', 'age', 'customer_value']
Colunas binárias preservadas em 0/1: ['complains', 'tariff_plan', 'status']


In [11]:
# Comparação descritiva usando somente o treino. Estes objetos são temporários;
# os transformadores dos modelos serão ajustados novamente dentro dos pipelines.
treino_standard = pd.DataFrame(
    StandardScaler().fit_transform(X_train[colunas_numericas]),
    columns=colunas_numericas,
)
treino_minmax = pd.DataFrame(
    MinMaxScaler().fit_transform(X_train[colunas_numericas]),
    columns=colunas_numericas,
)

comparacao_escalas = pd.DataFrame({
    "original_min": X_train[colunas_numericas].min(),
    "original_max": X_train[colunas_numericas].max(),
    "standard_media": treino_standard.mean(),
    "standard_desvio": treino_standard.std(ddof=0),
    "minmax_min": treino_minmax.min(),
    "minmax_max": treino_minmax.max(),
})

display(comparacao_escalas.round(4))

,original_min,original_max,standard_media,standard_desvio,minmax_min,minmax_max
call_failure,0.0,36.00,-0.0,1.0,0.0,1.0
subscription_length,3.0,47.00,0.0,1.0,0.0,1.0
charge_amount,0.0,10.00,0.0,1.0,0.0,1.0
seconds_of_use,0.0,17090.00,-0.0,1.0,0.0,1.0
frequency_of_use,0.0,255.00,0.0,1.0,0.0,1.0
frequency_of_sms,0.0,522.00,-0.0,1.0,0.0,1.0
distinct_called_numbers,0.0,97.00,-0.0,1.0,0.0,1.0
age,15.0,55.00,-0.0,1.0,0.0,1.0
customer_value,0.0,2165.28,-0.0,1.0,0.0,1.0
